# Kaggle smoke test (PLAN.md section 9, Gate 0 item 3)

Settings: Accelerator **GPU T4 x2**, Internet **on**, input: your private `repo.zip` dataset. Run top to bottom.
Each section ticks one checklist item and writes to `/kaggle/working/smoke_results.json`.
Rerun once with **GPU P100** for the P100 line (skip the vLLM section there; vLLM does not support P100).

In [ ]:
!nvidia-smi

In [ ]:
# Code comes from a PRIVATE Kaggle dataset holding repo.zip, made on your laptop with:
#   git archive --format=zip --add-virtual-file=COMMIT:$(git rev-parse HEAD) -o repo.zip HEAD
# Add it with "Add Input". Upload a new version of the dataset whenever the code changes.
import shutil
from pathlib import Path
src = next(Path("/kaggle/input").rglob("src/common/schemas.py")).parents[2]
shutil.rmtree("/tmp/repo", ignore_errors=True)
shutil.copytree(src, "/tmp/repo")
%cd /tmp/repo
!cat COMMIT

In [ ]:
!pip install -q -U bitsandbytes peft accelerate
!pip install -q -r requirements.txt

In [ ]:
import json, time, torch
from pathlib import Path
RESULTS = Path("/kaggle/working/smoke_results.json")
results = json.loads(RESULTS.read_text()) if RESULTS.exists() else {}
def save(**kw):
    results.update(kw); RESULTS.write_text(json.dumps(results, indent=2)); print(json.dumps(kw, indent=2))

MODEL = "Qwen/Qwen2.5-7B-Instruct"   # swap to compare candidates
GPU = torch.cuda.get_device_name(0)
save(gpu=GPU, gpu_count=torch.cuda.device_count(), torch=torch.__version__)

## 1. 4-bit load and one timed generation (`hf` backend)

In [ ]:
from src.models.generate import generate
from src.models.verify import build_prompt, parse_answer

prompt = build_prompt("The Lessee shall not carry on any business similar to that carried on at the Premises "
                      "within three kilometres of the Premises for two years after the expiry of this Lease.")
t = time.time(); generate(["Say OK."], MODEL, backend="hf", max_new_tokens=5); load_s = time.time() - t
t = time.time(); [out] = generate([prompt], MODEL, backend="hf", max_new_tokens=300); call_s = time.time() - t
t = time.time(); generate([prompt] * 8, MODEL, backend="hf", max_new_tokens=300, batch_size=8); batch_s = time.time() - t
print(out)
save(**{f"{GPU}/hf_4bit": {"model": MODEL, "load_s": round(load_s, 1), "seconds_per_call": round(call_s, 1),
        "seconds_per_call_batch8": round(batch_s / 8, 1), "parsed": parse_answer(out),
        "peak_gb": round(torch.cuda.max_memory_allocated() / 2**30, 2)}})

## 2. 20-step QLoRA, peak memory, checkpoint and resume

Dummy pairs only: this tests the machinery, not the model.

In [ ]:
import gc, subprocess
from src.models import generate as g
g._loaded.clear(); gc.collect(); torch.cuda.empty_cache()

def dummy_pairs(path, n=64, words=60):
    pair = {"pair_id": "D", "defect_type": "time_limit", "family": "TL-1", "form": "single", "definitions": [],
            "elements": {"affected_party": "tenant", "what_is_limited": "time_to_sue", "period_and_trigger": "3 months",
                         "effect_on_expiry": "claim_barred", "bank_guarantee": "no"}, "verdict": "void", "provision": "ICA-28",
            "clause": " ".join(["Any claim by the Lessee shall be barred unless a suit is filed within three months."] * (words // 15))}
    Path(path).write_text("".join(json.dumps(pair) + "\n" for _ in range(n)))

dummy_pairs("/tmp/dummy.jsonl")
OUT = "/kaggle/working/runs/smoke-qlora"
!python -m src.train.qlora --model {MODEL} --data /tmp/dummy.jsonl --out {OUT} --max-steps 20 --save-steps 10 --grad-accum 1
save(**{f"{GPU}/qlora_20_steps": json.loads(Path(OUT, "train_log.json").read_text())})

Resume check: **restart the session** (Run > Restart), rerun the two setup cells and the results cell, then this cell. It must start from `checkpoint-20`, not step 0.

In [ ]:
!python -m src.train.qlora --model {MODEL} --data /tmp/dummy.jsonl --out /kaggle/working/runs/smoke-qlora --max-steps 30 --save-steps 10 --grad-accum 1 2>&1 | grep -iE "resum|checkpoint|steps" | head
log = json.loads(Path("/kaggle/working/runs/smoke-qlora/train_log.json").read_text())
save(**{f"{GPU}/resume": {"ok": log["steps"] == 30, "steps": log["steps"]}})

## 3. Longest input that fits for QLoRA (aim: 1,024 tokens)

In [ ]:
dummy_pairs("/tmp/long.jsonl", n=8, words=3000)
fits = {}
for max_len in [1024, 1536, 2048, 3072]:
    out = f"/tmp/len-{max_len}"
    r = subprocess.run(["python", "-m", "src.train.qlora", "--model", MODEL, "--data", "/tmp/long.jsonl", "--out", out,
                        "--max-steps", "3", "--grad-accum", "1", "--max-len", str(max_len), "--save-steps", "1000"],
                       capture_output=True, text=True)
    fits[max_len] = json.loads(Path(out, "train_log.json").read_text())["peak_gpu_gb"] if r.returncode == 0 else "OOM/failed"
    print(max_len, fits[max_len])
    if r.returncode: print(r.stderr[-1500:]); break
save(**{f"{GPU}/qlora_max_len_peak_gb": fits})

## 4. vLLM timing (T4 only)

Installing vLLM may replace torch, so **restart the session first**, rerun the two setup cells, then run this section.

In [ ]:
!pip install -q vllm

In [ ]:
import json, time, torch
from pathlib import Path
from src.models.generate import generate
from src.models.verify import build_prompt
RESULTS = Path("/kaggle/working/smoke_results.json"); results = json.loads(RESULTS.read_text())
MODEL = "Qwen/Qwen2.5-7B-Instruct"
prompt = build_prompt("The Lessee shall not carry on any business similar to that carried on at the Premises "
                      "within three kilometres of the Premises for two years after the expiry of this Lease.")
kw = {"tensor_parallel_size": torch.cuda.device_count()}   # fp16 7B needs both T4s
t = time.time(); generate(["Say OK."], MODEL, backend="vllm", max_new_tokens=5, vllm_kw=kw); load_s = time.time() - t
t = time.time(); [out] = generate([prompt], MODEL, backend="vllm", max_new_tokens=300, vllm_kw=kw); call_s = time.time() - t
t = time.time(); generate([prompt] * 32, MODEL, backend="vllm", max_new_tokens=300, vllm_kw=kw); batch_s = time.time() - t
print(out)
results["vllm"] = {"model": MODEL, "vllm_kw": kw, "load_s": round(load_s, 1), "seconds_per_call": round(call_s, 1),
                   "seconds_per_call_batch32": round(batch_s / 32, 2)}
RESULTS.write_text(json.dumps(results, indent=2)); print(json.dumps(results, indent=2))

## 5. Background run

Click **Save Version > Save & Run All (Commit)** with the browser closed, then download `smoke_results.json` from the
version's Output tab. That ticks the last checklist item. Paste `smoke_results.json` into the mid-submission notes.